# Verification of the objective flow pattern criterion

Checks of the criterion version reported in the paper (`CRITERIA_VERSION`), run after `kde_plots.ipynb`. Nothing here changes the criterion: it reads its features and writes CSV files to `<version>/verification/`.

| Section | Output |
|---|---|
| Precision and recall per class | `precision_recall.csv` |
| Sampling rate (raw files) and record length | `sampling_per_run.csv`, `sampling_by_pattern.csv` |
| Liquid bridges and record length | `bridge_record_length.csv` |
| Pressure-gradient tolerance (±0.15 kPa, flow rates) | `pressure_bounds.csv` |
| Wall reference of the film estimate (0.985 mm ± 0.01 mm) | `wall_reference_sensitivity.csv` |
| Kilohertz runs downsampled to 100 Hz | `downsample_100hz.csv` |
| Friction model near the former laminar/turbulent switch (Re 1800-4000) | `friction_switch.csv` |
| Test-section pressure estimate (±10 kPa) | `ptp_sensitivity.csv` |
| Record duration by flow pattern and rate | `duration_by_pattern_rate.csv` |
| 100 Hz downsampling at every sampling phase | `downsample_100hz_phases.csv` |
| Undefined features and sentinels | `undefined_features.csv` |
| Baseline (`BASELINE_VERSION`) vs this version | `baseline_vs_revised.csv`, `baseline_vs_revised_summary.csv` |

In [1]:
import glob
import itertools
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import src.flow_pattern_criteria as fpc
from src.flow_pattern_criteria import (
    OBJECTIVE_PATTERNS,
    JL_MIN,
    balanced_accuracy,
    classify_frame,
    features_table,
    load_run,
    reference_pattern,
    run_features,
)
from src.pressure_features import pressure_features, two_phase_state, PRESSURE_FEATURES
from src.dpdz_correlations import friction_churchill, friction_f

CRITERIA_VERSION = 'objective_criteria_10'   # version reported in the paper
BASELINE_VERSION = 'objective_criteria_8'   # version of the previous manuscript

silver = project_root / 'data/silver_layer'
gold = project_root / 'data/gold_layer'
crit = gold / CRITERIA_VERSION
out = crit / 'verification'
out.mkdir(exist_ok=True)

features = pd.read_csv(crit / 'processed_v2/objective_flow_pattern.csv', index_col=0)
assert len(features) == 157

df_experimental = pd.read_pickle(silver / 'experimental_results.pkl')
df_eval = df_experimental[df_experimental['jl'] >= JL_MIN].copy()
df_eval['reduced_pattern'] = reference_pattern(df_eval['reduced_pattern'])
assert len(df_eval) == 157

VARIANT = {'path': silver / 'confocal_runs_processed_v2', 'raw_path': silver / 'confocal_runs'}


def metrics(visual, objective):
    ba, agreement = balanced_accuracy(visual, objective, OBJECTIVE_PATTERNS)
    return {'agreement': agreement, 'balanced_accuracy': ba, 'n_correct': int((np.asarray(visual) == np.asarray(objective)).sum())}


def with_pressure(df_features, df_pressure):
    """Feature table with the pressure features replaced by df_pressure."""
    df = df_features.drop(columns=[c for c in ['dpdz_homog', 'dpdz_l'] + PRESSURE_FEATURES if c in df_features.columns])
    return df.merge(df_pressure, on='run_id')


metrics(features['reduced_pattern'], features['objective_pattern'])

{'agreement': 0.8789808917197452,
 'balanced_accuracy': 0.8805009276437847,
 'n_correct': 138}

## Precision and recall per class

Recall: correctly identified runs / runs of the visual class. Precision: correctly identified runs / runs assigned to the class by the criterion.

In [2]:
visual, objective = features['reduced_pattern'], features['objective_pattern']
rows = []

for p in OBJECTIVE_PATTERNS:
    correct = int(((visual == p) & (objective == p)).sum())
    n_visual, n_objective = int((visual == p).sum()), int((objective == p).sum())
    rows.append({
        'pattern': p, 'n_visual': n_visual, 'n_objective': n_objective, 'correct': correct,
        'recall': correct / n_visual, 'precision': correct / n_objective if n_objective else np.nan,
    })

df_pr = pd.DataFrame(rows)
df_pr.to_csv(out / 'precision_recall.csv', index=False)
df_pr.round(3)

,pattern,n_visual,n_objective,correct,recall,precision
0,bubbly,11,11,11,1.000,1.000
1,plug,33,31,30,0.909,0.968
2,slug,25,23,18,0.720,0.783
3,churn,14,16,12,0.857,0.750
4,dispersed,20,14,14,0.700,1.000
5,wavy,44,52,43,0.977,0.827
6,annular,10,10,10,1.000,1.000


## Sampling rate and record length

The measuring rate is read from the raw confocal files (`Measuring rate` column, kHz; most frequent valid value, as the rows without signal carry the no-reading code), in the run order of `process_raw_confocal_data.ipynb`, and checked against the median time step of the treated runs. Record length is the treated record (after trimming, mixer-wave samples excluded).

In [3]:
# Same folders, headers and file order as process_raw_confocal_data.ipynb
FOLDERS = [
    ('2024_02_27', 12), ('2024_02_28', 12), ('2024_03_04', 12), ('2024_03_05', 12), ('2024_03_06', 12),
    ('2024_03_12', 12), ('2024_03_13', 12), ('2024_03_14', 12), ('2024_03_16', 14), ('2024_03_18', 14),
    ('2024_03_19', 14), ('2024_03_21', 14), ('2024_03_22', 14), ('2024_03_22_v2', 14), ('2024_03_23', 14),
    ('2024_03_25', 14), ('2024_03_25_v2', 14), ('2024_03_25_v3', 14), ('2024_03_26', 14),
]

rows = []
run_id = 1

for date, header in FOLDERS:
    for filename in glob.glob(str(project_root / 'data/bronze_layer/confocal' / date) + '/*.csv'):
        raw = pd.read_csv(filename, sep=';', decimal=',', header=header,
                          usecols=lambda c: c.startswith('Measuring rate'))
        # Rows without signal carry the no-reading code (2147483640) instead of the rate
        rate = raw.iloc[:, 0]
        rate = rate[(rate > 0) & (rate < 1000)]
        rows.append({'run_id': run_id, 'raw_file': f'{date}/{Path(filename).name}',
                     'rate_khz': rate.mode().iloc[0] if len(rate) else np.nan, 'rate_constant': rate.nunique() == 1})
        run_id += 1

df_rate = pd.DataFrame(rows)
assert len(df_rate) == 164

df_sampling = features[['run_id', 'reduced_pattern', 'objective_pattern', 'n_samples', 'duration_s',
                        'sampling_hz', 'low_sample_flag', 'bridge_count', 'bridge_freq_hz']].merge(df_rate, on='run_id')

# Raw rate vs median time step of the treated record
df_sampling['rate_mismatch'] = (df_sampling['sampling_hz'] / (1000 * df_sampling['rate_khz']) - 1).abs() > 0.02
df_sampling['rate_band'] = pd.cut(df_sampling['rate_khz'], [0, 0.5, 2.0, 10.0], labels=['0.1 kHz', '1.3-2 kHz', '> 2 kHz'])
df_sampling.to_csv(out / 'sampling_per_run.csv', index=False)

print('rate mismatches:', df_sampling.loc[df_sampling['rate_mismatch'], 'run_id'].tolist())
print('non-constant rate within a file:', df_sampling.loc[~df_sampling['rate_constant'], 'run_id'].tolist())
print(df_sampling['rate_band'].value_counts().sort_index())

df_by_pattern = df_sampling.groupby('reduced_pattern').agg(
    runs=('run_id', 'size'),
    at_0p1_khz=('rate_band', lambda s: int((s == '0.1 kHz').sum())),
    at_1p3_2_khz=('rate_band', lambda s: int((s == '1.3-2 kHz').sum())),
    above_2_khz=('rate_band', lambda s: int((s == '> 2 kHz').sum())),
    duration_median_s=('duration_s', 'median'),
    duration_min_s=('duration_s', 'min'),
    below_1000_samples=('low_sample_flag', 'sum'),
).reindex(OBJECTIVE_PATTERNS)
df_by_pattern.to_csv(out / 'sampling_by_pattern.csv')
df_by_pattern.round(1)

rate mismatches: [14, 15, 25, 26, 27, 35, 41, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58]
non-constant rate within a file: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67]
rate_band
0.1 kHz      98
1.3-2 kHz    50
> 2 kHz       9
Name: count, dtype: int64


,runs,at_0p1_khz,at_1p3_2_khz,above_2_khz,duration_median_s,duration_min_s,below_1000_samples
reduced_pattern,,,,,,,
bubbly,11,0,11,0,7.0,6.6,0
plug,33,23,10,0,17.3,4.5,2
slug,25,14,11,0,9.5,6.7,2
churn,14,9,2,3,10.9,2.8,1
dispersed,20,14,2,4,31.2,2.7,2
wavy,44,28,14,2,15.2,2.7,2
annular,10,10,0,0,44.8,3.7,1


## Liquid bridges and record length

A single liquid bridge in a record shorter than 10 s already gives $f_{bridge} \geq 0.1$ Hz. The table lists the runs whose intermittent/separated decision rests on a few bridges, and the disagreements with their rate and record length.

In [4]:
df_bridges = df_sampling.copy()
df_bridges['min_detectable_hz'] = 1 / df_bridges['duration_s']
df_bridges['one_bridge_is_intermittent'] = df_bridges['min_detectable_hz'] >= 0.1
df_bridges['few_bridges'] = df_bridges['bridge_count'].between(1, 2)
df_bridges['disagreement'] = df_bridges['reduced_pattern'] != df_bridges['objective_pattern']

cols = ['run_id', 'reduced_pattern', 'objective_pattern', 'rate_khz', 'n_samples', 'duration_s',
        'bridge_count', 'bridge_freq_hz', 'one_bridge_is_intermittent']
df_bridges.loc[df_bridges['few_bridges'] | df_bridges['disagreement'], cols].to_csv(out / 'bridge_record_length.csv', index=False)

print('records where one bridge is enough to be intermittent:', int(df_bridges['one_bridge_is_intermittent'].sum()))
print('runs with 1-2 bridges:', df_bridges.loc[df_bridges['few_bridges'], 'run_id'].tolist())
df_bridges.loc[df_bridges['disagreement'], cols].sort_values(['reduced_pattern', 'objective_pattern']).round(3)

records where one bridge is enough to be intermittent: 69
runs with 1-2 bridges: [23, 99, 103, 110]


,run_id,reduced_pattern,objective_pattern,rate_khz,n_samples,duration_s,bridge_count,bridge_freq_hz,one_bridge_is_intermittent
26,27,churn,wavy,1.3,10000,7.469,0,0.000,True
133,140,churn,wavy,0.1,2687,26.860,0,0.000,False
7,8,dispersed,churn,3.6,10000,2.838,0,0.000,True
8,9,dispersed,churn,3.6,10000,2.827,0,0.000,True
126,133,dispersed,churn,0.1,8207,82.060,0,0.000,False
127,134,dispersed,churn,0.1,1827,18.260,0,0.000,False
49,54,dispersed,slug,1.5,10000,6.833,3,0.439,True
50,55,dispersed,slug,1.4,10000,6.964,34,4.882,True
34,35,plug,slug,1.6,9991,6.103,19,3.113,True
35,36,plug,slug,1.6,9994,6.144,34,5.534,True


## Pressure-gradient tolerance

One-at-a-time (each tolerance alone, both signs) and simultaneous worst-case (all 27 sign combinations) bounds of $R_h$ and $\phi_l^2$ with the specified tolerance of the test-section differential pressure transducer (±0.15 kPa over $L$ = 0.75 m, i.e. ±200 Pa/m) and of the flow rates (liquid ±3 %, gas ±5 %, Table 1), taken one at a time and combined (all 27 sign combinations). This is a bound from the manufacturer and calibration tolerances, not a propagated statistical uncertainty.

In [5]:
DP_TOL = 150.0 / 0.75   # [Pa/m]
JL_TOL, JG_TOL = 0.03, 0.05

nominal = pressure_features(df_eval)
check = nominal.merge(features[['run_id'] + PRESSURE_FEATURES], on='run_id', suffixes=('', '_stored'))
assert np.allclose(check['ratio_homog'], check['ratio_homog_stored']) and np.allclose(check['phi2_l'], check['phi2_l_stored'])

cases = {}

for sdp, sjl, sjg in itertools.product((-1, 0, 1), repeat=3):
    d = df_eval.copy()
    d['dpdz_tp'] = d['dpdz_tp'] + sdp * DP_TOL
    d['jl'] = d['jl'] * (1 + sjl * JL_TOL)
    d['jg'] = d['jg'] * (1 + sjg * JG_TOL)
    pf = pressure_features(d)
    pred = classify_frame(with_pressure(features, pf))
    cases[(sdp, sjl, sjg)] = pf.set_index('run_id')[PRESSURE_FEATURES].assign(pattern=pd.Series(pred, index=features['run_id'].to_numpy()))

nominal = nominal.set_index('run_id')
bounds = pd.DataFrame(index=nominal.index)
bounds['dpdz_tp'] = df_eval.set_index('run_id')['dpdz_tp']
bounds['dp_tolerance_rel'] = DP_TOL / bounds['dpdz_tp']

GROUPS = {
    'dp_only': [k for k in cases if k[1] == k[2] == 0],
    'jl_only': [k for k in cases if k[0] == k[2] == 0],
    'jg_only': [k for k in cases if k[0] == k[1] == 0],
    'all': list(cases),
}
for name, keys in GROUPS.items():
    for feat in PRESSURE_FEATURES:
        values = pd.concat([cases[k][feat] for k in keys], axis=1)
        bounds[f'{feat}_min_{name}'] = values.min(axis=1)
        bounds[f'{feat}_max_{name}'] = values.max(axis=1)
    patterns = pd.concat([cases[k]['pattern'] for k in keys], axis=1)
    bounds[f'patterns_{name}'] = patterns.apply(lambda r: '/'.join(sorted(set(r))), axis=1)

bounds = bounds.join(features.set_index('run_id')[['reduced_pattern', 'objective_pattern'] + PRESSURE_FEATURES])
bounds.to_csv(out / 'pressure_bounds.csv')

print('relative dp tolerance (quantiles 0, 0.1, 0.5, 0.9, 1):', bounds['dp_tolerance_rel'].quantile([0, .1, .5, .9, 1]).round(3).tolist())
for name in GROUPS:
    changed = bounds[bounds[f'patterns_{name}'].str.contains('/')]
    print(f'runs whose class can change ({name}):', [(int(r), changed.loc[r, 'reduced_pattern'], changed.loc[r, f'patterns_{name}']) for r in changed.index])

bounds[bounds['patterns_all'].str.contains('/')][['reduced_pattern', 'objective_pattern', 'patterns_dp_only', 'patterns_all',
                                                   'ratio_homog', 'ratio_homog_min_all', 'ratio_homog_max_all',
                                                   'phi2_l', 'phi2_l_min_all', 'phi2_l_max_all']].round(3)

relative dp tolerance (quantiles 0, 0.1, 0.5, 0.9, 1): [0.004, 0.008, 0.059, 0.368, 1.105]
runs whose class can change (dp_only): [(91, 'slug', 'churn/wavy'), (116, 'churn', 'churn/wavy')]
runs whose class can change (jl_only): [(27, 'churn', 'churn/wavy'), (91, 'slug', 'churn/wavy'), (95, 'churn', 'churn/dispersed'), (116, 'churn', 'churn/wavy'), (136, 'dispersed', 'churn/dispersed')]
runs whose class can change (jg_only): [(27, 'churn', 'churn/wavy'), (91, 'slug', 'churn/wavy'), (116, 'churn', 'churn/wavy')]
runs whose class can change (all): [(21, 'wavy', 'churn/wavy'), (27, 'churn', 'churn/wavy'), (91, 'slug', 'churn/wavy'), (95, 'churn', 'churn/dispersed'), (116, 'churn', 'churn/wavy'), (136, 'dispersed', 'churn/dispersed'), (140, 'churn', 'churn/wavy')]


,reduced_pattern,objective_pattern,patterns_dp_only,patterns_all,ratio_homog,ratio_homog_min_all,ratio_homog_max_all,phi2_l,phi2_l_min_all,phi2_l_max_all
run_id,,,,,,,,,,
21,wavy,wavy,wavy,churn/wavy,0.926,0.851,1.011,18.569,17.357,19.890
27,churn,wavy,wavy,churn/wavy,0.982,0.902,1.071,11.602,10.839,12.434
91,slug,wavy,churn/wavy,churn/wavy,0.981,0.879,1.095,9.053,8.251,9.931
95,churn,churn,churn,churn/dispersed,1.265,1.161,1.381,5.786,5.390,6.219
116,churn,churn,churn/wavy,churn/wavy,1.023,0.932,1.125,9.663,8.945,10.447
136,dispersed,dispersed,dispersed,churn/dispersed,1.340,1.249,1.441,5.472,5.174,5.795
140,churn,wavy,wavy,churn/wavy,0.941,0.864,1.029,13.675,12.759,14.674


## Wall reference of the film estimate

The film below the detection limit is estimated from the excess of the first thickness over 0.985 mm. The empty-tube measurements (Table 2) vary by about ±0.01 mm in the mean bottom thickness between days, so the estimate is recomputed with 0.975 and 0.995 mm. Only the estimate of the criterion is changed; the treated second thickness (which also uses 0.985 mm) is kept.

In [6]:
pressure_cols = ['run_id', 'dpdz_homog', 'dpdz_l'] + PRESSURE_FEATURES
reference_cols = ['run_id', 'reduced_pattern']
rows = []

try:
    for t1_ref in (0.975, 0.985, 0.995):
        fpc.T1_THRESHOLD = t1_ref
        f = features_table(features['run_id'].tolist(), VARIANT['path'], raw_path=VARIANT['raw_path'])
        f = f.merge(features[reference_cols], on='run_id').merge(features[pressure_cols], on='run_id')
        f['objective_pattern'] = classify_frame(f)
        for _, r in f.iterrows():
            rows.append({'t1_ref_mm': t1_ref, 'run_id': r['run_id'], 'reduced_pattern': r['reduced_pattern'],
                         'objective_pattern': r['objective_pattern'], 'film_est_mean_mm': r['film_est_mean_mm'],
                         'film_est_std_mm': r['film_est_std_mm']})
finally:
    fpc.T1_THRESHOLD = 0.985

df_wall = pd.DataFrame(rows)
df_wall.to_csv(out / 'wall_reference_sensitivity.csv', index=False)

base = df_wall[df_wall['t1_ref_mm'] == 0.985].set_index('run_id')['objective_pattern']
assert (base == features.set_index('run_id')['objective_pattern']).all()

summary = df_wall.groupby(['t1_ref_mm', 'reduced_pattern'])['film_est_mean_mm'].median().unstack().reindex(columns=OBJECTIVE_PATTERNS)
for t1_ref, g in df_wall.groupby('t1_ref_mm'):
    changed = g.set_index('run_id')['objective_pattern'] != base
    m = metrics(g['reduced_pattern'], g['objective_pattern'])
    print(f't1_ref = {t1_ref}: class changes {changed.sum()} {changed[changed].index.tolist()}, '
          f'agreement {100 * m["agreement"]:.1f} %, BA {100 * m["balanced_accuracy"]:.1f} %')
summary.round(3)

t1_ref = 0.975: class changes 3 [1, 4, 37], agreement 86.0 %, BA 85.7 %
t1_ref = 0.985: class changes 0 [], agreement 87.9 %, BA 88.1 %
t1_ref = 0.995: class changes 0 [], agreement 87.9 %, BA 88.1 %


reduced_pattern,bubbly,plug,slug,churn,dispersed,wavy,annular
t1_ref_mm,,,,,,,
0.975,1.264,0.768,0.875,0.345,1.010,0.232,0.057
0.985,2.984,1.011,1.143,0.445,1.054,0.271,0.055
0.995,3.369,1.011,1.163,0.463,1.054,0.293,0.049


## Kilohertz runs downsampled to 100 Hz

Each run recorded at 1.3 kHz or more is reduced to the sample nearest to every 10 ms (no averaging) from the treated record *before* the mixer-wave removal (at 100 Hz the removal cannot act), and the features and classes are recomputed. The instrument at 100 Hz uses a longer exposure, which this procedure does not reproduce, so the result only isolates the effect of the time resolution on the features.

In [7]:
KHZ = features.loc[features['sampling_hz'] > 500, 'run_id'].tolist()
processed = silver / 'confocal_runs_processed'
rows = []

for run_id in KHZ:
    df = load_run(run_id, processed, raw_path=VARIANT['raw_path'])
    grid = np.arange(0.0, df['time'].iloc[-1], 0.01)
    idx = np.unique(np.clip(np.searchsorted(df['time'].to_numpy(), grid), 0, len(df) - 1))
    rows.append({'run_id': run_id, **run_features(df.iloc[idx].reset_index(drop=True))})

df_down = pd.DataFrame(rows).merge(features[reference_cols + ['objective_pattern']].rename(columns={'objective_pattern': 'objective_khz'}), on='run_id')
df_down = df_down.merge(features[pressure_cols], on='run_id')
df_down['objective_100hz'] = classify_frame(df_down)

keep = ['run_id', 'reduced_pattern', 'objective_khz', 'objective_100hz', 'bridge_freq_hz', 'bridge_mean_duration_s',
        'film_est_mean_mm', 'film_jump_frac', 'n_peaks']
df_down[keep].to_csv(out / 'downsample_100hz.csv', index=False)

changed = df_down[df_down['objective_khz'] != df_down['objective_100hz']]
print(f'{len(KHZ)} kHz runs; class changes at 100 Hz: {len(changed)}')
print('kHz   :', metrics(df_down['reduced_pattern'], df_down['objective_khz']))
print('100 Hz:', metrics(df_down['reduced_pattern'], df_down['objective_100hz']))
changed[keep].round(3)

59 kHz runs; class changes at 100 Hz: 9
kHz   : {'agreement': 0.8135593220338984, 'balanced_accuracy': 0.7616161616161617, 'n_correct': 48}
100 Hz: {'agreement': 0.7457627118644068, 'balanced_accuracy': 0.6914772727272727, 'n_correct': 44}


,run_id,reduced_pattern,objective_khz,objective_100hz,bridge_freq_hz,bridge_mean_duration_s,film_est_mean_mm,film_jump_frac,n_peaks
0,5,dispersed,dispersed,wavy,0.000,0.000,0.858,NaN,1
1,6,dispersed,dispersed,wavy,0.000,0.000,0.819,NaN,1
3,8,dispersed,churn,wavy,0.000,0.000,0.647,NaN,1
8,13,slug,slug,plug,7.397,0.027,0.895,0.194,2
10,15,slug,wavy,slug,0.411,0.017,0.882,0.285,1
31,36,plug,slug,plug,1.792,0.057,0.532,0.044,2
33,41,wavy,wavy,slug,0.134,0.020,1.121,0.055,1
38,46,wavy,wavy,slug,0.267,0.010,0.755,0.251,1
39,47,wavy,wavy,slug,0.404,0.010,0.620,0.182,1


## Friction model near the former laminar/turbulent switch

The previous model (56.91/Re below Re = 2300, Fang et al. above) jumps from f = 0.0248 to 0.0473 at Re = 2300. The Churchill (1977) correlation with the laminar-equivalent Reynolds number of Jones (1976) is continuous. Runs with $Re_{tp}$ or $Re_l$ between 1800 and 4000, and the classes with each friction model (same gas mass flux, at the meter density), isolate the effect of the friction model.

In [8]:
REF_RE = [500, 1000, 2000, 2299, 2300, 3000, 4000, 1e4, 3e4, 1e5]
unit = pd.DataFrame({
    'Re': REF_RE,
    'churchill_jones_square': [float(friction_churchill(r, po=56.91)) for r in REF_RE],
    'piecewise_square_fang': [friction_f(r, laminar_fre=56.91) for r in REF_RE],
    'laminar_56.91/Re': [56.91 / r for r in REF_RE],
    'churchill_round': [float(friction_churchill(r)) for r in REF_RE],
})
print(unit.round(5).to_string(index=False))

state = two_phase_state(df_eval).set_index('run_id')
pf_new = pressure_features(df_eval).set_index('run_id')
pf_old = pressure_features(df_eval, friction='piecewise').set_index('run_id')
switch = state[['Re_tp', 'Re_l']].join(pf_new[PRESSURE_FEATURES]).join(pf_old[PRESSURE_FEATURES], rsuffix='_piecewise')
switch['near_switch'] = switch['Re_tp'].between(1800, 4000) | switch['Re_l'].between(1800, 4000)
f_idx = features.set_index('run_id')
switch['pattern_churchill'] = classify_frame(with_pressure(features, pf_new.reset_index()))
switch['pattern_piecewise'] = classify_frame(with_pressure(features, pf_old.reset_index()))
switch = switch.join(f_idx[['reduced_pattern']])
switch.to_csv(out / 'friction_switch.csv')
print('runs with Re_tp in 1800-4000:', int(switch['Re_tp'].between(1800, 4000).sum()),
      '| Re_l in 1800-4000:', int(switch['Re_l'].between(1800, 4000).sum()))
diff = switch[switch['pattern_churchill'] != switch['pattern_piecewise']]
diff[['reduced_pattern', 'pattern_piecewise', 'pattern_churchill', 'Re_tp', 'Re_l', 'ratio_homog_piecewise', 'ratio_homog', 'phi2_l_piecewise', 'phi2_l']].round(3)

      Re  churchill_jones_square  piecewise_square_fang  laminar_56.91/Re  churchill_round
   500.0                 0.11382                0.11382           0.11382          0.12800
  1000.0                 0.05691                0.05691           0.05691          0.06400
  2000.0                 0.03027                0.02845           0.02845          0.03204
  2299.0                 0.03718                0.02475           0.02475          0.03083
  2300.0                 0.03721                0.04728           0.02474          0.03084
  3000.0                 0.04267                0.04353           0.01897          0.04297
  4000.0                 0.03914                0.03992           0.01423          0.04059
 10000.0                 0.03002                0.03090           0.00569          0.03100
 30000.0                 0.02275                0.02349           0.00190          0.02339
100000.0                 0.01744                0.01799           0.00057          0.01787

runs with Re_tp in 1800-4000: 29 | Re_l in 1800-4000: 31


,reduced_pattern,pattern_piecewise,pattern_churchill,Re_tp,Re_l,ratio_homog_piecewise,ratio_homog,phi2_l_piecewise,phi2_l
run_id,,,,,,,,,
116,churn,wavy,churn,9007.781,5782.177,0.995,1.023,9.432,9.663


## Test-section pressure estimate

$P_{tp} = P_{atm} + 3.5\,\Delta p_{tp}$ (regression from tests with an absolute transducer). Its effect is bounded by shifting $P_{tp}$ by ±10 kPa in the properties (gas density, viscosity) of all runs.

In [9]:
rows = []
for shift in (-10e3, 10e3):
    d = df_eval.copy()
    d['Ptp'] = d['Ptp'] + shift
    pf = pressure_features(d).set_index('run_id')
    pred = pd.Series(classify_frame(with_pressure(features, pf.reset_index())), index=features['run_id'].to_numpy())
    for run_id in features['run_id']:
        rows.append({'shift_Pa': shift, 'run_id': run_id,
                     'ratio_homog_change_pct': 100 * (pf.loc[run_id, 'ratio_homog'] / nominal.loc[run_id, 'ratio_homog'] - 1),
                     'phi2_l_change_pct': 100 * (pf.loc[run_id, 'phi2_l'] / nominal.loc[run_id, 'phi2_l'] - 1),
                     'pattern': pred[run_id], 'pattern_nominal': f_idx.loc[run_id, 'objective_pattern']})
ptp = pd.DataFrame(rows)
ptp.to_csv(out / 'ptp_sensitivity.csv', index=False)
print(ptp.groupby('shift_Pa')[['ratio_homog_change_pct', 'phi2_l_change_pct']].agg(['median', 'min', 'max']).round(2))
print('class changes:', ptp.loc[ptp['pattern'] != ptp['pattern_nominal'], ['shift_Pa', 'run_id', 'pattern_nominal', 'pattern']].values.tolist())

         ratio_homog_change_pct               phi2_l_change_pct          
                         median    min    max            median  min  max
shift_Pa                                                                 
-10000.0                  -7.35 -10.56  -1.14               0.0 -0.0  0.0
 10000.0                   7.08   0.97  10.54              -0.0 -0.0  0.0
class changes: [[-10000.0, 116, 'churn', 'wavy'], [10000.0, 27, 'wavy', 'churn'], [10000.0, 91, 'wavy', 'churn'], [10000.0, 140, 'wavy', 'churn']]


## Record duration and sample count by flow pattern and acquisition rate

In [10]:
dur = df_sampling.groupby(['reduced_pattern', 'rate_band'], observed=True).agg(
    runs=('run_id', 'size'), samples_median=('n_samples', 'median'), samples_min=('n_samples', 'min'),
    duration_median_s=('duration_s', 'median'), duration_min_s=('duration_s', 'min'), duration_max_s=('duration_s', 'max'),
).reset_index()
dur.to_csv(out / 'duration_by_pattern_rate.csv', index=False)
print(df_sampling.groupby('rate_band', observed=True)[['n_samples', 'duration_s']].describe().round(1).T)
dur.round(1)

rate_band         0.1 kHz  1.3-2 kHz  > 2 kHz
n_samples  count     98.0       50.0      9.0
           mean    5431.2     9857.6   9999.9
           std     3951.4      443.1      0.3
           min      375.0     7049.0   9999.0
           25%     1639.2     9956.5  10000.0
           50%     4325.5    10000.0  10000.0
           75%    10000.0    10000.0  10000.0
           max    10000.0    10000.0  10000.0
duration_s count     98.0       50.0      9.0
           mean      54.3        7.0      2.9
           std       39.5        0.6      0.3
           min        3.7        4.5      2.7
           25%       16.4        6.7      2.7
           50%       43.2        7.2      2.8
           75%      100.0        7.3      2.8
           max      100.0        7.6      3.6


,reduced_pattern,rate_band,runs,samples_median,samples_min,duration_median_s,duration_min_s,duration_max_s
0,annular,0.1 kHz,10,4476.5,375,44.8,3.7,100.0
1,bubbly,1.3-2 kHz,11,10000.0,10000,7.0,6.6,7.2
2,churn,0.1 kHz,9,1896.0,595,19.0,5.9,100.0
3,churn,1.3-2 kHz,2,9996.5,9993,7.5,7.5,7.5
4,churn,> 2 kHz,3,10000.0,10000,2.8,2.8,2.8
5,dispersed,0.1 kHz,14,7157.0,698,71.6,7.0,100.0
6,dispersed,1.3-2 kHz,2,10000.0,10000,6.9,6.8,7.0
7,dispersed,> 2 kHz,4,10000.0,10000,2.8,2.7,2.8
8,plug,0.1 kHz,23,9477.0,472,94.8,4.7,100.0
9,plug,1.3-2 kHz,10,10000.0,7049,6.2,4.5,6.7


## 100 Hz downsampling at every sampling phase

The kHz records are reduced to one sample every 10 ms starting at each possible offset (0 to k-1 samples, k = samples per 10 ms), from the treated record before the mixer-wave removal. Unchanged exposure and record length: this isolates the time resolution, not the effect of a longer 100 Hz acquisition.

In [11]:
rows = []
for run_id in KHZ:
    df = load_run(run_id, processed, raw_path=VARIANT['raw_path'])
    t = df['time'].to_numpy()
    k = int(round(0.01 / np.median(np.diff(t))))
    for offset in range(k):
        idx = np.arange(offset, len(df), k)
        feat = run_features(df.iloc[idx].reset_index(drop=True))
        rows.append({'run_id': run_id, 'offset': offset, 'k': k, **feat})
ds = pd.DataFrame(rows).merge(features[reference_cols + ['objective_pattern']].rename(columns={'objective_pattern': 'objective_khz'}), on='run_id')
ds = ds.merge(features[pressure_cols], on='run_id')
ds['objective_100hz'] = classify_frame(ds)
ds = ds.merge(df_sampling[['run_id', 'rate_khz']], on='run_id')
ds[['run_id', 'offset', 'k', 'rate_khz', 'reduced_pattern', 'objective_khz', 'objective_100hz']].to_csv(out / 'downsample_100hz_phases.csv', index=False)

per_run = ds.groupby('run_id').agg(rate_khz=('rate_khz', 'first'), visual=('reduced_pattern', 'first'), khz=('objective_khz', 'first'),
                                   offsets=('offset', 'size'),
                                   changed_offsets=('objective_100hz', lambda s: int((s != ds.loc[s.index, 'objective_khz']).sum())),
                                   classes_100hz=('objective_100hz', lambda s: '/'.join(sorted(set(s)))))
changed = per_run[per_run['changed_offsets'] > 0]
print(f'{len(KHZ)} kHz runs; runs changing class at least at one phase: {len(changed)}; at every phase: {int((per_run["changed_offsets"] == per_run["offsets"]).sum())}')
changes_by_phase = ds.assign(change=ds['objective_100hz'] != ds['objective_khz']).groupby('offset')['change'].sum()
print('class changes per phase offset (min-max):', int(changes_by_phase.min()), '-', int(changes_by_phase.max()))
changed

59 kHz runs; runs changing class at least at one phase: 16; at every phase: 2
class changes per phase offset (min-max): 1 - 13


,rate_khz,visual,khz,offsets,changed_offsets,classes_100hz
run_id,,,,,,
5,3.7,dispersed,dispersed,37,36,dispersed/wavy
6,3.7,dispersed,dispersed,37,35,dispersed/wavy
8,3.6,dispersed,churn,35,35,wavy
9,3.6,dispersed,churn,35,1,churn/dispersed
13,1.4,slug,slug,14,14,plug
15,1.4,slug,wavy,14,13,slug/wavy
36,1.6,plug,slug,16,7,plug/slug
37,2.8,wavy,wavy,28,7,annular/wavy
41,1.3,wavy,wavy,13,7,slug/wavy


## Undefined features and sentinels

How the implementation handles the cases without the signal needed for a feature (values used by the rules).

In [12]:
f = features
n_detected = (1 - f['f_undetected']) * f['n_samples']
undef = pd.DataFrame({
    'N_p = 0 because < 50 detected samples (not estimated)': [int((n_detected < 50).sum())],
    'N_p = 0 with >= 50 detected samples': [int(((n_detected >= 50) & (f['n_peaks'] == 0)).sum())],
    'J undefined (< 30 valid pairs)': [int(f['film_jump_frac'].isna().sum())],
    'S_b set to 0 (no bubble or film samples)': [int(((f['f_bubble'] + f['f_film']) == 0).sum())],
    't_bridge set to 0 (no liquid bridge)': [int((f['bridge_count'] == 0).sum())],
    'film mean/std undefined (< 5 values)': [int(f['film_est_mean_mm'].isna().sum())],
}).T.rename(columns={0: 'runs'})
undef.to_csv(out / 'undefined_features.csv')
print(f.loc[n_detected < 50, ['run_id', 'reduced_pattern', 'objective_pattern']].values.tolist())
undef

[[3, 'annular', 'annular'], [4, 'churn', 'churn'], [83, 'annular', 'annular'], [84, 'annular', 'annular'], [85, 'annular', 'annular'], [86, 'annular', 'annular'], [147, 'wavy', 'wavy'], [162, 'annular', 'annular'], [163, 'annular', 'annular']]


,runs
N_p = 0 because < 50 detected samples (not estimated),9
N_p = 0 with >= 50 detected samples,0
J undefined (< 30 valid pairs),18
S_b set to 0 (no bubble or film samples),1
t_bridge set to 0 (no liquid bridge),89
film mean/std undefined (< 5 values),0


## Baseline vs this version

Per-run pressure features and classes of `BASELINE_VERSION` (previous manuscript) and of `CRITERIA_VERSION`, and the summary metrics.

In [13]:
base = pd.read_csv(gold / BASELINE_VERSION / 'processed_v2/objective_flow_pattern.csv', index_col=0)
cols = ['run_id', 'reduced_pattern', 'dpdz_homog', 'dpdz_l', 'ratio_homog', 'phi2_l', 'objective_pattern']
comp = base[cols].merge(features[cols], on=['run_id', 'reduced_pattern'], suffixes=('_baseline', '_revised'))
for c in ['dpdz_homog', 'dpdz_l', 'ratio_homog', 'phi2_l']:
    comp[f'{c}_change_pct'] = 100 * (comp[f'{c}_revised'] / comp[f'{c}_baseline'] - 1)
comp['class_changed'] = comp['objective_pattern_baseline'] != comp['objective_pattern_revised']
comp.to_csv(out / 'baseline_vs_revised.csv', index=False)

summary = pd.DataFrame({
    BASELINE_VERSION: metrics(comp['reduced_pattern'], comp['objective_pattern_baseline']),
    CRITERIA_VERSION: metrics(comp['reduced_pattern'], comp['objective_pattern_revised']),
}).T
for p in OBJECTIVE_PATTERNS:
    for tag, col in ((BASELINE_VERSION, 'objective_pattern_baseline'), (CRITERIA_VERSION, 'objective_pattern_revised')):
        summary.loc[tag, f'n_{p}'] = int((comp[col] == p).sum())
summary.loc[:, 'class_changes'] = [0, int(comp['class_changed'].sum())]
summary.loc[:, 'pressure_features_changed_runs'] = [0, int((comp['phi2_l_change_pct'].abs() > 1e-6).sum())]
summary.to_csv(out / 'baseline_vs_revised_summary.csv')

print(comp.loc[comp['phi2_l_change_pct'].abs() > 1e-6, ['ratio_homog_change_pct', 'phi2_l_change_pct']].describe().round(2))
summary

       ratio_homog_change_pct  phi2_l_change_pct
count                  124.00             124.00
mean                    -6.15               1.01
std                      7.99               7.23
min                    -37.21             -32.59
25%                    -10.11               1.87
50%                     -3.32               2.66
75%                     -1.07               3.11
max                      2.85              17.53


,agreement,balanced_accuracy,n_correct,n_bubbly,n_plug,n_slug,n_churn,n_dispersed,n_wavy,n_annular,class_changes,pressure_features_changed_runs
objective_criteria_8,0.885350,0.893766,139.0,11.0,31.0,23.0,19.0,13.0,50.0,10.0,0,0
objective_criteria_10,0.878981,0.880501,138.0,11.0,31.0,23.0,16.0,14.0,52.0,10.0,3,124
